# Key-Value Databases — First Contact

The world's fastest notepad. Every piece of data has a key — you look it up instantly, no scanning, no joins. Redis keeps everything in RAM, which is why it answers in microseconds. It is not just strings — Redis has lists, sets, sorted sets, hashes, and streams built in. Think of it less as a database and more as a supercharged shared memory layer that your whole application stack can read and write atomically.

## What makes key-value stores different

- **In-memory** — data lives in RAM, not disk; sub-millisecond reads regardless of dataset size.
- **Simple data model** — a key maps to a value; no schema, no foreign keys, no query planner needed.
- **Rich types** — string, list, hash, set, sorted set, stream — each optimised for a specific access pattern.
- **When to use** — caching, session storage, leaderboards, rate limiting, pub/sub messaging, real-time counters.

In [7]:
from pathlib import Path
import sys

for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_redis_conn
import pandas as pd
import json

r = get_redis_conn()
print("Connected. Redis ping:", r.ping())
print("Keys in DB:", r.dbsize())

Connected. Redis ping: True
Keys in DB: 10008


## 5 Redis patterns against the telemetry data

In [8]:
# Pattern 1 — Hash (endpoint lookup cache)
# Redis HASH = one key holds multiple fields, like a lightweight row.
# Endpoints were cached as hashes during seed. Key format: endpoint:{endpoint_id}

sample_keys = r.keys("endpoint:*")[:3]
for key in sample_keys:
    data = r.hgetall(key)
    print(f"\nKey: {key}")
    for field, value in data.items():
        print(f"  {field}: {value}")


Key: endpoint:709235c1-0f7f-46b2-a413-5c6e4b9cc899
  hostname: srv-04138.citi.internal
  datacenter: NYC1
  environment: dev
  service_type: monitor
  ip_address: 172.26.221.228
  os: windows
  status: active

Key: endpoint:b531a1eb-c3aa-4e46-9fb6-79b08fd74070
  hostname: srv-05556.citi.internal
  datacenter: LON1
  environment: staging
  service_type: worker
  ip_address: 192.168.42.240
  os: linux
  status: active

Key: endpoint:3c4702b8-bc65-4d95-85a2-c4d9c139fbb0
  hostname: srv-07018.citi.internal
  datacenter: LON1
  environment: prod
  service_type: cache
  ip_address: 172.25.213.11
  os: linux
  status: active


In [9]:
# Pattern 2 — DBSIZE and sorted set (alert count leaderboard)
# How many endpoint hash keys exist?
endpoint_keys = r.keys("endpoint:*")
print(f"Total endpoint hash keys: {len(endpoint_keys):,}")

# Check alert_counts sorted set
alert_key = r.exists("alert_counts")
print(f"alert_counts sorted set exists: {bool(alert_key)}")

# Top 5 endpoints by alert count (sorted set — O(log N))
top_alerted = r.zrevrange("alert_counts", 0, 4, withscores=True)
print("\nTop 5 most alerted endpoints (from sorted set):")
for endpoint_id, count in top_alerted:
    print(f"  {endpoint_id}: {int(count)} alerts")

Total endpoint hash keys: 10,001
alert_counts sorted set exists: True

Top 5 most alerted endpoints (from sorted set):
  3dce0c5f-b03a-4076-9241-6822cebc3c33: 12 alerts
  537ce2da-d8ab-4d4d-b516-2007e641af45: 11 alerts
  80a56561-f8b0-4cc7-9a2f-584668c983bc: 10 alerts
  77bc3422-b331-45a0-9732-631c2f7faa6c: 10 alerts
  5ded145d-815f-4b3c-8ccb-515950c78589: 10 alerts


In [10]:
# Pattern 3 — Counter (atomic INCR)
# Redis INCR is atomic — safe under concurrent writers, no locks needed.
# Simulate counting 100 incoming alerts split by severity.

import random

r.delete("live:alert_counter")
for sev in ["critical", "high", "medium", "low"]:
    r.delete(f"live:alerts_by_severity:{sev}")

severities = ["critical", "high", "medium", "low"]
for _ in range(100):
    r.incr("live:alert_counter")
    r.incr(f"live:alerts_by_severity:{random.choice(severities)}")

total = r.get("live:alert_counter")
print(f"Total alerts processed: {total}")
for sev in severities:
    count = r.get(f"live:alerts_by_severity:{sev}") or 0
    print(f"  {sev}: {count}")

Total alerts processed: 100
  critical: 17
  high: 30
  medium: 28
  low: 25


In [11]:
# Pattern 4 — TTL (cache-aside with expiry)
# Store an expensive query result with a 60-second TTL.
# This is the cache-aside pattern: check cache first, hit DB on miss, set TTL to control staleness.

import time

cache_key = "cache:endpoint:summary:NYC1"

summary = {
    "datacenter": "NYC1",
    "total_endpoints": 2489,
    "active": 1876,
    "cached_at": time.strftime("%Y-%m-%dT%H:%M:%S")
}

r.setex(cache_key, 60, json.dumps(summary))  # store with 60s TTL
print(f"Cached with 60s TTL: {cache_key}")

cached = r.get(cache_key)
data = json.loads(cached)
print(f"Retrieved: {data}")

ttl = r.ttl(cache_key)
print(f"TTL remaining: {ttl} seconds")

Cached with 60s TTL: cache:endpoint:summary:NYC1
Retrieved: {'datacenter': 'NYC1', 'total_endpoints': 2489, 'active': 1876, 'cached_at': '2026-03-23T21:45:57'}
TTL remaining: 60 seconds


In [12]:
# Pattern 5 — List (recent alerts queue, capped at 10)
# Redis LIST supports push/pop from both ends.
# LPUSH + LTRIM = a fixed-length recent-events log.

r.delete("recent:critical_alerts")

for i in range(15):
    alert = json.dumps({
        "id": f"alert_{i:03d}",
        "host": f"srv-{random.randint(1, 9999):05d}.citi.internal",
        "msg": "CPU exceeded 90%"
    })
    r.lpush("recent:critical_alerts", alert)
    r.ltrim("recent:critical_alerts", 0, 9)  # keep only last 10

queue_len = r.llen("recent:critical_alerts")
print(f"Recent alerts queue length: {queue_len} (capped at 10)")

items = r.lrange("recent:critical_alerts", 0, -1)
print("\nQueue contents (newest first):")
for item in items:
    print(f"  {json.loads(item)}")

Recent alerts queue length: 10 (capped at 10)

Queue contents (newest first):
  {'id': 'alert_014', 'host': 'srv-06167.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_013', 'host': 'srv-05586.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_012', 'host': 'srv-06845.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_011', 'host': 'srv-06238.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_010', 'host': 'srv-08046.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_009', 'host': 'srv-08448.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_008', 'host': 'srv-02456.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_007', 'host': 'srv-04231.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_006', 'host': 'srv-06493.citi.internal', 'msg': 'CPU exceeded 90%'}
  {'id': 'alert_005', 'host': 'srv-00892.citi.internal', 'msg': 'CPU exceeded 90%'}


## Redis data types — quick reference

| Type | Commands | Use case |
|------|----------|----------|
| String | `SET` / `GET` / `INCR` | Simple cache, counters, flags |
| Hash | `HSET` / `HGETALL` / `HGET` | Object cache (endpoint details) |
| List | `LPUSH` / `LRANGE` / `LTRIM` | Event queue, recent items log |
| Set | `SADD` / `SMEMBERS` / `SISMEMBER` | Unique tags, membership checks |
| Sorted Set | `ZADD` / `ZREVRANGE` / `ZSCORE` | Leaderboards, alert rankings |
| Stream | `XADD` / `XREAD` | Real-time event streaming, CDC |
| TTL | `SETEX` / `EXPIRE` / `TTL` | Any key with automatic expiry |

## Key observations

- **Sub-millisecond** — all 5 patterns ran in RAM with no disk I/O; latency is bounded by network RTT, not storage.
- **Atomic operations** — `INCR` is guaranteed atomic even under thousands of concurrent writers; no locks, no race conditions for counters.
- **TTL is first-class** — every key can have an expiry; Redis evicts automatically when it fires. Cache invalidation is built in.
- **Not a replacement for Postgres** — Redis has no query language, no joins, no durable persistence by default (AOF/RDB are optional). It complements relational, it does not replace it.
- **Cache-aside pattern (Pattern 4)** is the most common DE use case: check Redis first → if miss, query Postgres → store result in Redis with TTL → next caller hits cache. Repeat.